In [13]:
import os, glob, shutil
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split,Subset
from torchvision import transforms

import torch, torch.nn as nn
from torchvision import models

In [14]:
data_dir = "./UTKFace" 

In [15]:

# src = "UTKFace"
# dst = "UTKFace_sorted"
# os.makedirs(f"{dst}/male", exist_ok=True)
# os.makedirs(f"{dst}/female", exist_ok=True)

# counts = {"male": 0, "female": 0, "skipped": 0}

# for path in glob.glob(os.path.join(src, "**", "*.jpg"), recursive=True):
#     name = os.path.basename(path)
#     try:
#         gender = int(name.split("_")[1])      # age_gender_race_datetime
#     except (IndexError, ValueError):
#         counts["skipped"] += 1                # malformed filename
#         continue

#     if gender == 0:
#         shutil.copy(path, f"{dst}/male/{name}")
#         counts["male"] += 1
#     elif gender == 1:
#         shutil.copy(path, f"{dst}/female/{name}")
#         counts["female"] += 1
#     else:
#         counts["skipped"] += 1

# print(counts)

In [16]:

class UTKFaceGender(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.samples = []
        # recursive glob, in case images are inside a nested subfolder
        for p in glob.glob(os.path.join(root, "**", "*.jpg"), recursive=True):
            parts = os.path.basename(p).split("_")
            try:
                gender = int(parts[1])
                if gender in (0, 1):
                    self.samples.append((p, gender))
            except (IndexError, ValueError):
                continue   # a few UTKFace files have malformed names; skip them

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


In [17]:
train_tfm = transforms.Compose([
    transforms.Resize((144, 144)),
    transforms.RandomCrop(128),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),   
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_tfm = transforms.Compose([
    transforms.Resize((144, 144)),
    transforms.CenterCrop(128),                 
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),

])

In [18]:
train_full = UTKFaceGender(data_dir, transform=train_tfm)
test_full  = UTKFaceGender(data_dir, transform=test_tfm)

# ---- Split indices once ----
n = len(train_full)
g = torch.Generator().manual_seed(42)
perm = torch.randperm(n, generator=g).tolist()
n_train = int(0.8 * n)


In [19]:

train_ds = Subset(train_full, perm[:n_train])   # augmented
test_ds  = Subset(test_full,  perm[n_train:])   # clean

# ---- DataLoaders ----

In [20]:
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,
                          num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False,
                          num_workers=0)

In [21]:
# import time
# t = time.time()
# for i in range(20):
#     train_ds[i]
# print(f"20 images: {time.time()-t:.1f}s")

In [22]:
x, y = next(iter(train_loader))
print(x.shape, y.shape)    # [64, 3, 128, 128], [64]
print(len(train_ds), len(test_ds))

torch.Size([64, 3, 128, 128]) torch.Size([64])
18966 4742


In [23]:

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


device: cuda


In [24]:

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 2)   # 2 classes: male / female
model = model.to(device)

In [25]:


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

In [26]:
def run_epoch(loader, train=True):
    model.train(train)
    total_loss, correct, count = 0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            count += x.size(0)
    return total_loss / count, correct / count

In [27]:

EPOCHS = 10
best_acc = 0
for epoch in range(EPOCHS):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    te_loss, te_acc = run_epoch(test_loader, train=False)
    scheduler.step()
    print(f"Epoch {epoch+1}/{EPOCHS} | train loss {tr_loss:.4f} acc {tr_acc:.4f} "
          f"| test loss {te_loss:.4f} acc {te_acc:.4f}")
    if te_acc > best_acc:
        best_acc = te_acc
        torch.save(model.state_dict(), "gender_resnet18.pt")

Epoch 1/10 | train loss 0.3294 acc 0.8502 | test loss 0.2204 acc 0.9110
Epoch 2/10 | train loss 0.2308 acc 0.9009 | test loss 0.2020 acc 0.9156
Epoch 3/10 | train loss 0.2009 acc 0.9159 | test loss 0.1847 acc 0.9281
Epoch 4/10 | train loss 0.1787 acc 0.9260 | test loss 0.1899 acc 0.9256
Epoch 5/10 | train loss 0.1619 acc 0.9351 | test loss 0.1814 acc 0.9323
Epoch 6/10 | train loss 0.1421 acc 0.9419 | test loss 0.1762 acc 0.9353
Epoch 7/10 | train loss 0.1249 acc 0.9499 | test loss 0.1820 acc 0.9321
Epoch 8/10 | train loss 0.1156 acc 0.9545 | test loss 0.1781 acc 0.9367
Epoch 9/10 | train loss 0.0970 acc 0.9627 | test loss 0.1762 acc 0.9384
Epoch 10/10 | train loss 0.0920 acc 0.9637 | test loss 0.1806 acc 0.9386
